In [2]:
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.applications import EfficientNetB0

IMG_SIZE = 224
BATCH_SIZE = 64

In [3]:
dataset_name = "stanford_dogs"
(ds_train, ds_test), ds_info = tfds.load(
    dataset_name, split=["train", "test"], with_info=True, as_supervised=True
)
NUM_CLASSES = ds_info.features["label"].num_classes

In [4]:
size = (IMG_SIZE, IMG_SIZE)
ds_train = ds_train.map(lambda image, label: (tf.image.resize(image, size), label))
ds_test = ds_test.map(lambda image, label: (tf.image.resize(image, size), label))

In [5]:
img_augmentation_layers = [
    layers.RandomRotation(factor=0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomFlip(),
    layers.RandomContrast(factor=0.1)
]

def img_augmentation(images):
    for layer in img_augmentation_layers:
        images = layer(images)
    return images

In [6]:
# one-hot / categical encoding
def input_preprocess_train(image, label):
    image = img_augmentation(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def input_preprocess_test(image, label):
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

ds_train = ds_train.map(input_preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.batch(batch_size=BATCH_SIZE, drop_remainder=True)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.map(input_preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(batch_size=BATCH_SIZE, drop_remainder=True)

In [7]:
import tensorflow as tf
model = tf.keras.models.load_model("./saves/stanford_dogs_effcientnet_b0.keras")

In [8]:
# 1 epoch 10분 정도
epochs = 10
hist = model.fit(ds_train, epochs=epochs, validation_data=ds_test)

Epoch 1/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 553s 3s/step - accuracy: 0.2000 - loss: 3.2023 - val_accuracy: 0.1219 - val_loss: 3.8893
Epoch 2/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 491s 3s/step - accuracy: 0.2120 - loss: 3.1177 - val_accuracy: 0.1410 - val_loss: 3.7417
Epoch 3/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 493s 3s/step - accuracy: 0.2259 - loss: 3.0470 - val_accuracy: 0.1305 - val_loss: 3.8803
Epoch 4/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 489s 3s/step - accuracy: 0.2501 - loss: 2.9475 - val_accuracy: 0.1532 - val_loss: 3.6985
Epoch 5/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 493s 3s/step - accuracy: 0.2664 - loss: 2.8564 - val_accuracy: 0.1277 - val_loss: 4.0204
Epoch 6/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 488s 3s/step - accuracy: 0.2787 - loss: 2.7841 - val_accuracy: 0.1574 - val_loss: 3.7513
Epoch 7/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 489s 3s/step - accuracy: 0.2965 - loss: 2.6963 - val_accuracy: 0.1602 - val_loss: 3.8154
Epoch 8/10
187/187 ━━━━━━━━━━━━━━━━━━━━ 488s 3s/step - accuracy: 0.3154 - loss: 2.6181 - val_accu

In [10]:
model.save("./saves/stanford_dogs_effcientnet_b0.keras")